# Phase 6 · Step 3 — Ridge Regression (Layer 3)

*Portfolio notebook for Portfolio Project 3 (Inflation Prediction and Economic Signal Analysis).*

This notebook narrates the Layer 3 Ridge regression contribution to the three-layer modelling architecture (ARIMA → VAR → **Ridge** per D-004). Every Step 3 decision (D-064..D-074) is narrated with rationale and the audit CSV it cites; all heavy computation lives in `scripts/phase6_step3_*.py` and in `src.modelling_utils` v0.4.2 — this notebook is a thin narrative layer.

**Step 3 delivers:**

- High-dimensional regularised CPI forecasting on the Phase 4 full superset (50–53 features per country)
- An independent L2-regularisation-based cross-lens confirmation of the N3 Japan Isolation narrative (7th lens)
- Phillips Methodology Trilogy → Quadrilogy extension with the stationary-form Ridge base-feature lens (Germany only)
- USA dual-form resolution for N2 narrative with Ridge first_diff ↔ VAR IRF cross-lens consistency
- Ridge vs VAR OOS forecast comparison: 12/16 cells Ridge relative-wins, 2/16 beats random-walk naive absolutely

All visualisations are regenerated from pre-computed audit CSVs; Run All completes in ≈ 30 seconds.


## Setup

Convention identical to `06_arima_baseline.ipynb` and `07_var_model.ipynb`: CSVs under `data/documentation/`, figures under `outputs/figures/phase6_step3_*.png`. `src/` at v0.4.2 per D-074 — no Ridge fit call inside this notebook; all model-fitting is in `scripts/phase6_step3_*.py`.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.ticker import LogLocator

# src v0.4.2 — Phase 6 Step 3 conventions
from src import (
    __version__ as SRC_VERSION,
    TRAIN_END, TEST_START,
    HORIZONS_PHASE7,
    ALPHA_GRID_DEFAULT,
    CATEGORY_ORDER,
    VAR_MASE_D060,
)

print(f"src version: {SRC_VERSION}")
assert SRC_VERSION >= "0.4.2", f"This notebook requires src >= 0.4.2 (got {SRC_VERSION})"

DOC_DIR = PROJECT_ROOT / "data" / "documentation"
FIG_DIR = PROJECT_ROOT / "outputs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Consistent country colours across Phase 6 notebooks
COLORS = {"USA": "#1f77b4", "JAPAN": "#d62728", "UK": "#2ca02c", "GERMANY": "#9467bd"}
FORM_LS = {"primary": "-", "first_diff_secondary": "--"}

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 120, "font.size": 10,
    "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False,
})

def load_csv(name: str) -> pd.DataFrame:
    # Read an audit CSV from data/documentation/.
    path = DOC_DIR / name
    if not path.exists():
        raise FileNotFoundError(
            f"Required audit CSV missing: {path.relative_to(PROJECT_ROOT)}. "
            f"Run the corresponding scripts/phase6_step3_*.py to regenerate."
        )
    return pd.read_csv(path)

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"Train / test split: .. {TRAIN_END.date()}  |  {TEST_START.date()} ..")
print(f"Horizons for Phase 7 DM: {HORIZONS_PHASE7}")


## 1. Context and Pipeline Overview

### 1.1 Why Ridge after ARIMA and VAR?

Phase 4 closed with per-country feature matrices of 50–53 columns each (5 base + 20 lag + 20 rolling + 5–8 regime interaction). D-040 deliberately deferred feature selection to Phase 6 under the reasoning that different Phase 6 models would select differently: ARIMA uses only CPI, VAR uses the 5-variable endogenous block, Ridge uses everything under L2 regularisation.

Phase 6 Step 1 (ARIMA, D-048 / D-049) and Step 2 (VAR, D-050..D-062, D-063) are complete. Step 3's task is the remaining Layer 3 deliverable: high-dimensional regularised regression that handles the multicollinearity Phase 4 preserved as a superset.

Ridge also closes a methodological gap: ARIMA is univariate-dynamic, VAR is multivariate-dynamic, Ridge is **multivariate-static** — a static regression treating all lag / rolling / regime information as a single flat feature vector. This bridges back to the ML methodology familiar from P2 (Portfolio Project 2) and ensures the project's modelling layers span the three canonical forecast paradigms.


### 1.2 Step 3 decision map (D-064..D-074)

| ID | Scope | Sub-step |
|---|---|---|
| D-064 | Full-superset scope + CPI target + USA dual-form | S1 |
| D-065 | α log-grid + TimeSeriesSplit CV + Pipeline leakage-guard | S2 |
| D-066 | Japan α extension + N3 septuple Ridge-lens | S2b |
| D-067 | Standardised coefs + 5-fold stability + Phillips quadrilogy | S3 |
| D-068 | OOS walk-forward direct-h + shared α + matched origins | S4 |
| D-069 | Regime-interaction zero-information (methodology finding) | S3 |
| D-070 | Ridge-vs-VAR relative-win, absolute-difficulty positioning | S5 |
| D-071 | USA dual-form resolved — first_diff preferred for N2 | S3/S4/S5b |
| D-072 | N3 septuple cross-lens formalisation (synthetic) | S5 |
| D-073 | Step 3 closeout — analytical freeze | — |
| D-074 | `src/modelling_utils` v0.4.2 promotion (this notebook uses it) | — |

Full rationale: `ProjectDriven.md` D-064..D-074 block.


### 1.3 Seven sub-steps and audit CSV trace

| Sub-step | Scratch script | Output audit CSVs |
|---|---|---|
| S1 | `phase6_step3_s1_data_preparation.py` | 3 CSVs (matrix / categories / target summaries) |
| S2 | `phase6_step3_s2_alpha_cv.py` | 2 CSVs (cv_scores / alpha_selection) |
| S2b | `phase6_step3_s2b_japan_grid_extension.py` | 2 CSVs (jpn_cv_scores / jpn_alpha_selection) |
| S3 | `phase6_step3_s3_coefficients.py` | 3 CSVs (coefficients / top_features / category_contribution) |
| S4 | `phase6_step3_s4_oos_forecast.py` | 3 CSVs (forecasts / metrics / cpi_summary) |
| S5 | `phase6_step3_s5_narrative_consolidation.py` | 3 CSVs (country_narrative / ridge_vs_var / statements) |
| S5b | `phase6_step3_s5b_narrative_correction.py` | 1 new CSV (phillips_lens) + 1 overwrite |

All 15 audit CSVs are under `data/documentation/phase6_step3_*.csv`.


## 2. Data Preparation — D-064

Ridge inputs at Step 3 are **five (country, form) combinations**: four primary-form matrices (USA yoy_pct, Japan first_diff, UK log_diff_pct, Germany first_diff per D-031) and one USA first_diff secondary matrix to complete the dual-form contest pre-committed in D-048 / D-062.

The USA secondary matrix is built by temporarily overriding `REGISTRY_OVERRIDES[('USA', 'CPI')]` to `first_diff` before running `build_country_features('USA')`, ensuring CPI-derived lag, rolling, and interaction columns are form-consistent end-to-end. This logic lives in `src.modelling_utils.build_usa_first_diff_features` (v0.4.2, D-074).

Train / test split follows D-005: `TRAIN_END = 2019-12-01`, `TEST_START = 2020-01-01`, constants exported from `src.modelling_utils`. Held identical to ARIMA Step 1 and VAR Step 2 S6 to enable Phase 7 Diebold-Mariano paired comparison.


In [ ]:
s1_matrix = load_csv("phase6_step3_s1_feature_matrix_summary.csv")
s1_cats   = load_csv("phase6_step3_s1_feature_categories.csv")
s1_target = load_csv("phase6_step3_s1_target_summary.csv")

# Merge for compact display
disp = s1_matrix.merge(
    s1_cats[["country", "form", "base", "lag", "rolling",
             "split", "period", "interaction"]],
    on=["country", "form"],
)[[
    "country", "form", "n_features_X", "n_train", "n_test",
    "first_valid_date", "last_valid_date",
    "base", "lag", "rolling", "split", "period", "interaction",
]]
disp


**Observations from the matrix table:**

- All four primary combinations share the Phase 4 structural pattern (4 base + 20 lag + 20 rolling features) plus country-specific regime structure (3 splits + 2 periods + 0–3 interactions). Total 49–52 features per country.
- USA's primary `yoy_pct` loses 11 additional months at the joint-valid start (2003-01 vs 2002-02 for the other three) due to the 12-month overlap of the year-over-year transform — the D-031 trade-off first flagged at Phase 5.
- USA's first_diff secondary matrix matches Japan / UK / Germany's joint-valid start (2002-02), gaining 11 observations of training data. This is a side effect of the form choice, not an artefact: first_diff truncates 1 leading observation vs yoy_pct's 12.

**Target dispersion** (from `phase6_step3_s1_target_summary.csv`): the 2020+ test window has approximately 2× the train-window standard deviation for all countries — reflecting the COVID deflation and 2022 energy-shock regime.


## 3. α Selection Protocol — D-065

Each of the five (country, form) combinations receives an α* selected by expanding-window walk-forward cross-validation:

- **Grid**: `ALPHA_GRID_DEFAULT = np.logspace(-3, 3, 13)` = {0.001, 0.00316, ..., 1000}, constant exported from `src.modelling_utils` per D-074.
- **CV**: `sklearn.model_selection.TimeSeriesSplit(n_splits=5)` restricted to the train window (≤ 2019-12). 2020+ is held out for S4 walk-forward OOS evaluation.
- **Standardisation**: `StandardScaler` fitted per fold's train split inside a `Pipeline` (built via `make_ridge_pipeline()` per D-065 / D-074) — no cross-fold leakage.
- **Selection rule**: argmin of mean validation MSE across 5 folds.

The 1-SE alternative α is recorded for parsimony sensitivity but is not used downstream per D-065.


In [ ]:
from src import load_selected_alphas

# S2 + S2b merged selection (Japan primary overridden by S2b)
alphas = load_selected_alphas()

# Tabular display
sel = pd.DataFrame([
    {"country": c, "form": f, "selected_alpha": a, "log10_alpha": np.log10(a)}
    for (c, f), a in sorted(alphas.items())
])
sel["log10_alpha"] = sel["log10_alpha"].round(2)
sel["selected_alpha"] = sel["selected_alpha"].round(4)
sel


In [ ]:
# Figure 1 — α path per (country, form)
s2_scores = load_csv("phase6_step3_s2_cv_scores.csv")
s2_sel = load_csv("phase6_step3_s2_alpha_selection.csv")

fig, ax = plt.subplots(figsize=(10, 5.5))

for (country, form), grp in s2_scores.groupby(["country", "form"]):
    path = grp.groupby("alpha")["val_mse"].mean().sort_index()
    ax.plot(path.index, path.values,
            linestyle=FORM_LS[form], color=COLORS[country],
            linewidth=1.8, marker="o", markersize=4,
            label=f"{country} · {form}", alpha=0.85)

    sel_row = s2_sel[(s2_sel["country"] == country) & (s2_sel["form"] == form)].iloc[0]
    ax.scatter(sel_row["selected_alpha"], sel_row["cv_val_mse_mean"],
               s=150, color=COLORS[country], edgecolor="black",
               zorder=10, marker="*")

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("α (log scale)")
ax.set_ylabel("Mean validation MSE across 5 folds (log scale)")
ax.set_title(
    "Figure 1 · Ridge α path per (country, form) — 13-point log-grid + TSS(5)",
    loc="left",
)
ax.legend(loc="best", frameon=False, fontsize=9)
ax.grid(which="both", alpha=0.25)

plt.tight_layout()
plt.savefig(FIG_DIR / "phase6_step3_fig1_alpha_path.png", bbox_inches="tight")
plt.show()


**Reading Figure 1:**

- Four of five combinations show a clear U-shape with interior minima at moderate α (log10 ∈ [+1.0, +2.0]): USA primary / USA first_diff / UK / GERMANY.
- **Japan primary is monotonically decreasing through α = 1000** (upper boundary of the initial grid). This is the diagnostic trigger for S2b's grid extension.
- The 316× α range from USA primary (α* = 10) to Japan (α* = 3162 post-S2b) is the largest cross-country regularisation-intensity difference in the project — the first quantitative fingerprint of the N3 Japan Isolation narrative at the Ridge layer.


## 4. Japan α Grid Extension — D-066

S2 flagged Japan primary at α = 1000 (upper boundary). Following the D-048 Stage (b) boundary-sensitivity protocol, S2b extends the Japan grid to `np.logspace(3, 6, 7)` — log10 α ∈ {3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0}.

The extended grid also computes a **theoretical intercept-only bound**: the validation MSE that a `ŷ_{t+h} = mean(y_train_fold)` predictor achieves under the same TimeSeriesSplit folds. This is the Ridge α → ∞ limit — the number Ridge converges toward when L2 completely shrinks all coefficients to zero.


In [ ]:
# Figure 2 — Japan extended α path + intercept-only saturation bound
s2b_scores = load_csv("phase6_step3_s2b_japan_cv_scores.csv")

path_ext = s2b_scores.groupby("alpha")["val_mse"].mean().sort_index()

# S2 original grid for Japan primary
s2_jpn = s2_scores[
    (s2_scores["country"] == "JAPAN") & (s2_scores["form"] == "primary")
].groupby("alpha")["val_mse"].mean().sort_index()

# Intercept-only bound: path_ext at α → ∞ (α = 1e6 approximates it closely)
intercept_bound = path_ext.values[-1]

# α* from S2b
alpha_star = float(alphas[("JAPAN", "primary")])

fig, ax = plt.subplots(figsize=(9, 5.5))

ax.plot(s2_jpn.index, s2_jpn.values, "o-",
        color=COLORS["JAPAN"], alpha=0.40, markersize=5,
        label="S2 initial grid (log10 α ∈ [−3, +3])")
ax.plot(path_ext.index, path_ext.values, "o-",
        color=COLORS["JAPAN"], alpha=1.0, markersize=7, linewidth=2,
        label="S2b extended grid (log10 α ∈ [+3, +6])")

ax.axhline(intercept_bound, color="black", linestyle=":", linewidth=1.2,
           label=f"Intercept-only bound ({intercept_bound:.4f})")

ax.axvline(alpha_star, color="black", linestyle="--", linewidth=0.8, alpha=0.5)
ax.scatter(alpha_star, path_ext.loc[alpha_star], s=200,
           color=COLORS["JAPAN"], edgecolor="black", zorder=10, marker="*",
           label=f"α* = {alpha_star:.0f} (log10 = +{np.log10(alpha_star):.1f})")

ax.set_xscale("log")
ax.set_xlabel("α (log scale)")
ax.set_ylabel("Mean validation MSE across 5 folds")
ax.set_title(
    "Figure 2 · JAPAN primary — α saturation at the intercept-only bound (D-066)",
    loc="left",
)
ax.legend(loc="best", frameon=False, fontsize=9)
ax.grid(which="both", alpha=0.25)

plt.tight_layout()
plt.savefig(FIG_DIR / "phase6_step3_fig2_japan_alpha_extension.png", bbox_inches="tight")
plt.show()


**N3 Japan Isolation — Ridge-lens confirmation (D-066):**

The extended grid reveals:

- **α* = 3162** at log10 = +3.5 (interior minimum)
- val_MSE = 0.0854, intercept-only bound = 0.0888
- Gain from the "optimal" Ridge fit over the intercept-only predictor: **0.0034 absolute, 3.8 % relative**
- Across the 49-feature Japan matrix, Ridge with carefully-tuned L2 improves on "just predict the mean" by less than 1/16 of the fold-level MSE standard deviation — statistically indistinguishable from zero.

This is the **7th independent lens on N3 Japan Isolation** (after ACF at D-044, ARIMA at D-049, VAR lag selection at D-050, Granger at D-052, IRF at D-056, FEVD at D-058 / D-059). Each lens uses fundamentally different mathematics; all seven converge on the same conclusion.

**D-072 formalises this as septuple confirmation.**


## 5. Coefficient Stability and Feature Importance — D-067

At the S2 / S2b-selected α*, each combination is fitted:

1. Once on the full 2000–2019 training window → primary coefficient magnitudes.
2. Five times across `TimeSeriesSplit(n_splits=5)` folds → coefficient stability envelope.

Coefficients are extracted from the `Pipeline`'s internal `Ridge` step, which operates in **standardised feature space** — magnitudes are directly comparable across features, countries, and forms without further rescaling. The `sign_stable` indicator flags features whose sign matches across the full-train fit and all 5 folds; this is the Ridge analogue of "Bonferroni-corrected significance" for portfolio interpretation.


In [ ]:
# Overall magnitude summary per combination
s3_coef = load_csv("phase6_step3_s3_ridge_coefficients.csv")

summary = s3_coef.groupby(["country", "form"]).agg(
    n_features=("feature_name", "count"),
    max_abs_coef=("abs_coef_full", "max"),
    sum_abs_coef=("abs_coef_full", "sum"),
    sign_stable_count=("sign_stable", "sum"),
).reset_index()

# Ratio column vs Japan primary (for N3 quantification)
jpn_max = float(
    summary[(summary["country"] == "JAPAN") & (summary["form"] == "primary")]
    ["max_abs_coef"].iloc[0]
)
summary["ratio_max_vs_JPN"] = (summary["max_abs_coef"] / jpn_max).round(1)

summary.round(4)


In [ ]:
# Figure 3 — |coef_full_train| boxplot per combination (log scale)
order = ["JAPAN-primary", "UK-primary", "GERMANY-primary",
         "USA-first_diff_secondary", "USA-primary"]
data_by_label = {}
for (country, form), grp in s3_coef.groupby(["country", "form"]):
    label = f"{country}-{form}"
    vals = grp["abs_coef_full"].values
    vals = vals[vals > 0]  # drop exact-zero interaction columns (D-069)
    data_by_label[label] = vals

fig, ax = plt.subplots(figsize=(10, 5))
positions = list(range(len(order)))
vals_order = [data_by_label[lbl] for lbl in order]

bp = ax.boxplot(vals_order, positions=positions, widths=0.55,
                patch_artist=True,
                medianprops=dict(color="black", linewidth=1.5))
color_map = {
    "JAPAN-primary": COLORS["JAPAN"],
    "UK-primary": COLORS["UK"],
    "GERMANY-primary": COLORS["GERMANY"],
    "USA-first_diff_secondary": COLORS["USA"],
    "USA-primary": COLORS["USA"],
}
for patch, label in zip(bp["boxes"], order):
    patch.set_facecolor(color_map[label])
    patch.set_alpha(0.55)
    if "first_diff" in label:
        patch.set_hatch("//")

ax.set_yscale("log")
ax.set_xticks(positions)
ax.set_xticklabels(order, rotation=20, ha="right", fontsize=9)
ax.set_ylabel("|coef_full_train| (log scale, standardised space)")
ax.set_title(
    "Figure 3 · Coefficient magnitude stratification — Japan dwarfed by 9.9–71.4×",
    loc="left",
)
ax.grid(axis="y", which="both", alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / "phase6_step3_fig3_coef_magnitude.png", bbox_inches="tight")
plt.show()


**Quantitative N3 fingerprint (D-067):**

- Japan primary max|coef| = **0.0100**
- Other countries' max|coef| range: 0.099 (UK) to 0.714 (USA primary)
- **Ratio Japan vs rest: 9.9× to 71.4×**
- This is independent of D-066's α-based evidence; two Ridge-layer fingerprints for N3.

**On the `sign_stable_count` column:** Japan has the lowest count (25 out of 49) because its coefficients are so close to zero that fold-to-fold noise flips their signs. USA primary has the highest (28/52) because its signal-to-noise at α = 10 is more favourable.


In [ ]:
# Figure 4 — Top-5 features per combination
s3_top = load_csv("phase6_step3_s3_top_features.csv")
top5 = s3_top[s3_top["rank"] <= 5].copy()

fig, axes = plt.subplots(1, 5, figsize=(16, 5), sharey=False)

combos_ordered = [
    ("USA", "primary"),
    ("JAPAN", "primary"),
    ("UK", "primary"),
    ("GERMANY", "primary"),
    ("USA", "first_diff_secondary"),
]

for ax, (country, form) in zip(axes, combos_ordered):
    sub = top5[(top5["country"] == country) & (top5["form"] == form)].sort_values("rank")
    y_pos = np.arange(len(sub))[::-1]
    colors_bar = [COLORS[country] if s else "lightgray"
                  for s in sub["sign_stable"].values]

    ax.barh(y_pos, sub["coef_full_train"].values, color=colors_bar,
            edgecolor="black", alpha=0.85)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(
        [f.replace(f"{country}_", "") for f in sub["feature_name"].values],
        fontsize=8,
    )
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("coef (standardised)", fontsize=9)
    ax.set_title(f"{country}\n{form}", fontsize=9, loc="center")
    ax.grid(axis="x", alpha=0.3)

fig.suptitle(
    "Figure 4 · Top-5 features per combination — greyed bars indicate sign-unstable",
    y=1.02, x=0.5, fontsize=11,
)
plt.tight_layout()
plt.savefig(FIG_DIR / "phase6_step3_fig4_top_features.png", bbox_inches="tight")
plt.show()


**Top-5 country patterns (Figure 4):**

- **USA primary (yoy_pct)** — dominated by CPI auto-features (`CPI_roll3_mean` rank 1, `CPI_lag1` rank 2). POLICY_RATE absent through rank 5 — D-071 resolution evidence.
- **JAPAN primary** — all five coefs near 0.003–0.010 magnitude; no feature stands out. Visual confirmation of D-067's magnitude stratification finding.
- **UK primary** — `CPI_lag12` rank 1 (seasonal), `CPI_roll3_mean` rank 2. `POLICY_RATE` as a base feature enters at rank 5.
- **GERMANY primary** — CPI auto-features + **`UNEMPLOYMENT` base at rank 4**, coef −0.040, sign-stable. This triggers the Phillips Quadrilogy in §6.
- **USA first_diff secondary** — `POLICY_RATE_lag3` rank 2 coef −0.136; `POLICY_RATE_lag12` rank 4 coef +0.095. The first_diff form surfaces monetary transmission that yoy_pct hides.


## 6. Phillips Methodology Quadrilogy — D-067 (extends D-057 Trilogy)

D-057 recorded three mutually compatible but visually different Phillips-Curve manifestations:

1. Classical negative slope in level form (Phase 5 D-043)
2. Invisibility under stationary correlation (D-046)
3. Positive-sign in stationary VAR IRF (D-057)

S5b contributes a fourth lens: **stationary-form Ridge base-feature coefficient**, restricted to:

- `category == 'base'` (not lag, not rolling)
- `feature_name == f'{country}_UNEMPLOYMENT'` (exact suffix, contemporaneous)
- `rank <= 5` (top-5 by |coef_full_train|)
- `coef_full_train < 0` (classical Phillips sign)
- `sign_stable == True`

Only **GERMANY** survives all four criteria.


In [ ]:
# Figure 5 — Phillips Quadrilogy visualisation
phillips = load_csv("phase6_step3_s5b_phillips_base_feature_lens.csv")

# Lens × country matrix (project narrative claims)
lens_matrix = pd.DataFrame(
    {
        "Level (D-043)":           ["✓", "—", "✓", "—"],
        "Stat. corr (D-046)":      ["—", "—", "—", "—"],
        "VAR IRF (D-057)":         ["✓", "—", "✓", "—"],
        "Ridge base coef (D-067)": ["—", "—", "—", "✓"],
    },
    index=["USA", "JAPAN", "UK", "GERMANY"],
)
print("Phillips Methodology Quadrilogy — lens × country:\n")
print(lens_matrix.to_string())
print()

# Bar chart of each country's contemporaneous UNEMPLOYMENT coef
fig, ax = plt.subplots(figsize=(8, 4.5))
order_c = ["USA", "JAPAN", "UK", "GERMANY"]
coefs = [float(phillips[phillips["country"] == c]["coef_full_train"].iloc[0]) for c in order_c]
ranks = [int(phillips[phillips["country"] == c]["rank_abs_full"].iloc[0]) for c in order_c]
stable = [bool(phillips[phillips["country"] == c]["sign_stable"].iloc[0]) for c in order_c]
active = [bool(phillips[phillips["country"] == c]["phillips_lens_active"].iloc[0]) for c in order_c]

colors_bar = [COLORS[c] if a else "lightgray" for c, a in zip(order_c, active)]
edge = ["black" if s else "gray" for s in stable]
lw = [2.0 if a else 1.0 for a in active]
bars = ax.bar(order_c, coefs, color=colors_bar, edgecolor=edge, linewidth=lw)

for bar, r, s, c in zip(bars, ranks, stable, coefs):
    ax.text(bar.get_x() + bar.get_width() / 2, c - 0.003,
            f"rank {r}\n{'stable' if s else 'unstable'}",
            ha="center", va="top", fontsize=8)

ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("coef_full_train  ({country}_UNEMPLOYMENT, base)")
ax.set_title(
    "Figure 5 · Phillips Quadrilogy — contemporaneous UNEMPLOYMENT Ridge coef",
    loc="left", fontsize=11,
)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / "phase6_step3_fig5_phillips_quadrilogy.png", bbox_inches="tight")
plt.show()


**Quadrilogy reading:**

- **Germany** (rank 4, coef −0.040, sign-stable): **only country Phillips-active on the Ridge lens**. D-067's fourth-lens contribution.
- USA (rank 39, coef −0.019, sign-unstable): Ridge does not elevate contemporaneous UNEMPLOYMENT to interpretable status. USA's Phillips Curve appears on level (D-043) and VAR IRF (D-057) lenses but not Ridge.
- UK (rank 19, coef −0.007, sign-stable but outside top-5): D-057 VAR IRF lens surfaced UK's positive-sign anomaly, but Ridge base-feature lens does not.
- Japan (rank 27, coef −0.001, sign-unstable): effectively zero at the Ridge lens — consistent with N3 isolation.

**The meta-finding is lens-dependence**: no country is Phillips-active on all four lenses simultaneously. The classical "Phillips Curve" is real but emerges differently across analytical frames. D-057's Trilogy is formally extended to **Quadrilogy**.


## 7. Out-of-Sample Forecasts — D-068

For each combination and each horizon h ∈ `HORIZONS_PHASE7`, S4 performs direct multi-step walk-forward forecasting:

- Training pairs: `(X_s, y_{s+h})` for s ≤ origin − h
- α held at the S2 / S2b-selected value (shared across horizons per D-068 — mirrors D-050 VAR lag-order philosophy)
- Origin range via `compute_walk_forward_origins()` (D-074): `[2020-01, last_obs − 12 months]` so every horizon is evaluable at every origin (matches VAR S6 for paired DM)

Baseline for MASE and RMSE-ratio: random-walk `ŷ_{t+h} = y_t`.


In [ ]:
# Figure 6 — Ridge OOS MASE heatmap
s4_metrics = load_csv("phase6_step3_s4_ridge_oos_metrics.csv")

pivot_mase = s4_metrics.pivot_table(
    index=["country", "form"], columns="horizon", values="mase"
).round(3)

row_order = [
    ("USA", "primary"),
    ("USA", "first_diff_secondary"),
    ("JAPAN", "primary"),
    ("UK", "primary"),
    ("GERMANY", "primary"),
]
pivot_mase = pivot_mase.reindex(row_order)

fig, ax = plt.subplots(figsize=(8, 4.5))
data = pivot_mase.values
vmax = 5.0  # cap colormap so extreme USA cells don't wash out others
im = ax.imshow(data, aspect="auto", cmap="RdYlGn_r", vmin=0.0, vmax=vmax, origin="upper")

for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        val = data[i, j]
        color = "white" if val > 2.5 or val < 0.5 else "black"
        ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                color=color, fontsize=10,
                fontweight="bold" if val < 1.0 else "normal")

ax.set_xticks(range(len(pivot_mase.columns)))
ax.set_xticklabels([f"h={h}" for h in pivot_mase.columns])
ax.set_yticks(range(len(pivot_mase.index)))
ax.set_yticklabels([f"{c}·{f}" for c, f in pivot_mase.index], fontsize=9)
ax.set_xlabel("Forecast horizon (months)")
ax.set_title(
    "Figure 6 · Ridge OOS MASE — bold < 1.0 beats random-walk naive",
    loc="left", fontsize=11,
)
plt.colorbar(im, ax=ax, shrink=0.8, label="MASE (colour-capped at 5)")

plt.tight_layout()
plt.savefig(FIG_DIR / "phase6_step3_fig6_oos_mase_heatmap.png", bbox_inches="tight")
plt.show()


In [ ]:
# Figure 7 — USA dual-form OOS forecast comparison (D-071)
s4_fc = load_csv("phase6_step3_s4_ridge_oos_forecasts.csv")

fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True)
horizons_plot = [1, 3, 6, 12]

for ax, h in zip(axes.flat, horizons_plot):
    for form, ls in [("primary", "-"), ("first_diff_secondary", "--")]:
        sub = s4_fc[
            (s4_fc["country"] == "USA")
            & (s4_fc["form"] == form)
            & (s4_fc["horizon"] == h)
        ].sort_values("target_date")
        dates = pd.to_datetime(sub["target_date"])

        if form == "primary":
            ax.plot(dates, sub["actual"].values, color="black",
                    linewidth=1.2, label="actual")
        ax.plot(dates, sub["forecast"].values,
                linestyle=ls, color=COLORS["USA"], linewidth=1.5, alpha=0.85,
                label=f"Ridge {form}")

    ax.set_title(f"h = {h} months", fontsize=10)
    ax.grid(alpha=0.3)
    if h == 1:
        ax.legend(loc="best", frameon=False, fontsize=8)

fig.suptitle(
    "Figure 7 · USA dual-form OOS forecasts — first_diff vs yoy_pct (D-071)",
    y=1.00, x=0.5, fontsize=11,
)
for ax in axes[-1]:
    ax.set_xlabel("target date")
plt.tight_layout()
plt.savefig(FIG_DIR / "phase6_step3_fig7_usa_dual_form.png", bbox_inches="tight")
plt.show()


**OOS findings (Figures 6, 7):**

- **2/16 cells beat random-walk naive** (MASE < 1.0): JAPAN h=1 (0.98), UK h=3 (0.97). All other cells produce forecasts worse than simply predicting last month's value. This is the D-070 absolute-difficulty finding — 2020+ is hostile to any model at Layer 3 depth.
- **USA dual-form resolved for N2 narrative (D-071)**: first_diff secondary outperforms yoy_pct primary at all four horizons by 39 %–85 % on MASE. The yoy_pct form's systematic positive bias (+3.4 to +4.2 at h=6, h=12) reflects training-mean anchoring (USA train_mean = 2.10 %) against 2022 target values of 8 %+.
- Japan's near-MASE-1.0 across all four horizons is the D-072 septuple signature: neither Ridge nor VAR (D-060) extracts meaningful multivariate information for Japan.


## 8. Ridge vs VAR Cross-Layer Comparison — D-070

The 16-cell comparison (4 countries × 4 horizons × primary form) joins Ridge S4 MASE against VAR S6 MASE. The VAR values are sourced from `VAR_MASE_D060` in `src.modelling_utils` (D-074 — hardcoded from D-060 at AIC-selected p per country).

- **Ridge wins 12/16**: GERMANY 4/4, UK 4/4, USA 4/4, JAPAN 0/4
- Japan's VAR 4/4 wins are all small margins (≤ 18.7 %) and both layers close to MASE = 1.0
- UK h=12 is Ridge's most dramatic win: **77× MASE reduction** (VAR 79.07 → Ridge 1.02). The VAR outlier is documented at D-061; Ridge's L2 absorbs the COVID-origin shock without outlier generation.


In [ ]:
# Figure 8 — Ridge vs VAR comparison
s5_rvv = load_csv("phase6_step3_s5_ridge_vs_var_mase.csv")

fig, axes = plt.subplots(1, 2, figsize=(13, 5),
                         gridspec_kw={"width_ratios": [1.3, 1.0]})

# LEFT — grouped bar, 4 countries × 4 horizons, log scale
ax = axes[0]
country_order = ["GERMANY", "JAPAN", "UK", "USA"]
horizons_plot = [1, 3, 6, 12]
bar_width = 0.09
x_base = np.arange(len(horizons_plot))

for i, country in enumerate(country_order):
    sub = s5_rvv[s5_rvv["country"] == country].sort_values("horizon")
    x_pos = x_base + i * 0.22
    ax.bar(x_pos - 0.05, sub["ridge_mase"].values, width=bar_width,
           color=COLORS[country], alpha=1.0, edgecolor="black", linewidth=0.4)
    ax.bar(x_pos + 0.05, sub["var_mase_d060"].values, width=bar_width,
           color=COLORS[country], alpha=0.45, edgecolor="black",
           linewidth=0.4, hatch="//")

ax.set_yscale("log")
ax.axhline(1.0, color="black", linestyle=":", linewidth=1.0)
ax.set_xticks(x_base + 0.33)
ax.set_xticklabels([f"h={h}" for h in horizons_plot])
ax.set_ylabel("MASE (log scale)")
ax.set_title("Figure 8a · Ridge (solid) vs VAR (hatched) — 16 cells",
             loc="left", fontsize=10)
ax.grid(axis="y", which="both", alpha=0.3)

legend_elements = [Patch(facecolor=COLORS[c], label=c) for c in country_order] + [
    Patch(facecolor="gray", alpha=1.0, label="Ridge"),
    Patch(facecolor="gray", hatch="//", alpha=0.5, label="VAR (D-060)"),
    plt.Line2D([], [], color="black", linestyle=":", label="naive MASE=1"),
]
ax.legend(handles=legend_elements, loc="upper left", fontsize=8, frameon=False)

# RIGHT — winner tally
ax = axes[1]
winner_counts = s5_rvv["winner"].value_counts()
colors_pie = {"Ridge": "#2ca02c", "VAR": "#d62728", "Tie": "gray"}

ax.pie(
    winner_counts.values,
    labels=winner_counts.index,
    colors=[colors_pie[w] for w in winner_counts.index],
    autopct=lambda pct: f"{int(round(pct * len(s5_rvv) / 100))} cells\n({pct:.0f}%)",
    startangle=90,
    wedgeprops={"edgecolor": "white", "linewidth": 2},
    textprops={"fontsize": 11},
)
ax.set_title(
    "Figure 8b · Winner aggregation\n(12/16 Ridge vs VAR; Japan only VAR wins)",
    loc="center", fontsize=10,
)

plt.tight_layout()
plt.savefig(FIG_DIR / "phase6_step3_fig8_ridge_vs_var.png", bbox_inches="tight")
plt.show()


**Two-layered positioning (D-070):**

- **Relative layer**: Ridge 12/16 wins over VAR. This is the portfolio's primary forecasting claim — Layer 3 regularisation produces materially better OOS forecasts than Layer 2 inference-primary VAR in three of four countries.
- **Absolute layer**: Both Ridge and VAR mostly lose to random-walk naive. Layer 3 does not solve the 2020+ forecasting problem; it improves on Layer 2's attempt. Phase 7 DM will evaluate whether ARIMA (Layer 1) closes the absolute gap at short horizons where univariate dynamics dominate.


## 9. Phase 6 Step 3 — Seven Signature Findings

1. **N3 Japan Isolation SEPTUPLE-confirmed (D-072)** — seven independent lenses: ACF (D-044), ARIMA (D-049), VAR lag (D-050), Granger (D-052), IRF (D-056), FEVD (D-058 / D-059), **Ridge α + coef (D-066 / D-067)**. Ridge contribution: α* = 3162 is 32–316× larger than other countries; max|coef| = 0.0100 is 9.9–71.6× smaller.

2. **Ridge 12/16 over VAR (D-070)** — GERMANY / UK / USA 4/4 each; JAPAN 0/4 all small-margin. UK h=12 shows 77× MASE reduction as L2 absorbs the D-061 COVID-origin VAR outlier.

3. **Ridge forecast-improved, naive-non-dominant (D-070)** — 2/16 cells beat naive at Layer 3; portfolio records Ridge as cross-layer-improved rather than unconditionally dominant.

4. **Phillips Methodology Quadrilogy (D-067 extends D-057)** — Ridge base-feature lens adds fourth perspective on Phillips Curve; GERMANY only country Phillips-active on this lens.

5. **USA dual-form resolved for N2 (D-071)** — first_diff preferred: top-5 surfaces POLICY_RATE_lag3 (−0.136) cross-lens matching VAR IRF peak (−0.149 at h=4, D-056). MASE 39–85 % lower than yoy_pct.

6. **Regime-interaction zero-information (D-069)** — 5/6 D-030-gated interactions are zero across the 2000–2019 train window; Ridge correctly shrinks them. Methodology transparency rather than bug.

7. **Coefficient-magnitude stratification (D-067)** — Japan 0.010 << UK 0.099 ≈ Germany 0.154 << USA_fd 0.356 < USA_primary 0.714. Second independent Ridge-layer N3 fingerprint beyond α.


### 9.1 D-072 — N3 Japan Isolation septuple cross-lens matrix

| # | Lens | Decision | Japan-specific finding |
|:-:|---|:-:|---|
| 1 | ACF[12] | D-044 | Near-white-noise; weakest of 4 countries |
| 2 | ARIMA AIC | D-049 | Interior min at lag 5; only non-boundary |
| 3 | VAR AIC ext | D-050 | Interior min at lag 5 stable |
| 4 | Granger | D-052 | 0/4 CPI receivers Bonferroni-significant |
| 5 | VAR IRF | D-056 | 4/4 CPI-response CIs straddle zero |
| 6 | VAR FEVD | D-058 / D-059 | 92% self-share plateau at h=24 |
| 7 | **Ridge** | **D-066 / D-067** | **α*=3162 (32–316× others); max&#124;coef&#124;=0.010 (9.9–71.6× smaller); OOS MASE h=1 = 0.978 marginal beat** |

Seven independent mathematical objects — correlation, likelihood, hypothesis testing, dynamic response, variance share, penalised coefficient magnitude — all converge on the same Japan-specific isolation conclusion. This cross-methodological robustness is the project's most intellectually defensible finding and the centerpiece of the Phase 8 findings.md narrative.


### 9.2 Methodology notes

- **Regime-interaction zero-information (D-069)**: the 5 post-2020 interaction columns are structurally zero across the 2000–2019 train window. Ridge's L2 penalty correctly drives their coefficients to zero — not a bug but a consequence of D-005 (train=2000–2019) intersected with D-036 (regime dummies emit for all D-030-gated pairs).

- **α-shared-across-horizons (D-068)**: Ridge α* was selected once per combination in S2 / S2b, then held constant across all four forecast horizons in S4. Per-horizon α retuning was rejected as an asymmetric advantage over VAR's D-050 shared-lag policy.

- **Standardised coefficient space (D-067)**: All coefficient magnitudes in this notebook are in standardised feature space (i.e., after `StandardScaler` z-scoring). Raw-space interpretations would require multiplying each coefficient by its feature standard deviation — meaningful for individual features but not for cross-feature comparison.

- **`src.modelling_utils` v0.4.2 (D-074)**: 6 Step-3-duplicated items were promoted to `src/` following the D-063 4×-duplication rule. This notebook demonstrates the promoted API by importing `TRAIN_END`, `TEST_START`, `HORIZONS_PHASE7`, `ALPHA_GRID_DEFAULT`, `VAR_MASE_D060`, and `load_selected_alphas()` directly from `src`.


## 10. Phase 6 Closure — Handoff State

With this notebook, Phase 6's three-layer modelling architecture is analytically complete:

| Layer | Notebook | Status |
|---|---|:-:|
| 1 — ARIMA | `06_arima_baseline.ipynb` | ✅ D-048, D-049 |
| 2 — VAR | `07_var_model.ipynb` | Pending assembly; D-050..D-063 |
| 3 — Ridge | `08_ridge_regression.ipynb` | **this notebook — D-064..D-074** |

**Remaining Phase 6 closure tasks:**

1. `07_var_model.ipynb` assembly — consolidate D-050..D-063 narratives from 9 scratch scripts, same thin-narrative-layer style as this notebook and `06_arima_baseline.ipynb`.
2. `phase6_step2_summary.md` amendment — update "Seven Signature Findings" item #1 from SEXTUPLE-confirmed to SEPTUPLE-confirmed per D-072.
3. `src/` v0.5.0 module assembly decision — per D-073 / D-074 deferral; re-evaluate at Phase 6 closure whether `modelling_utils` absorbs full model wrappers or a new `src/models/` submodule is introduced.
4. `phase6_summary.md` — top-level Phase 6 consolidation (ARIMA + VAR + Ridge) for Project Knowledge.
5. `README.md` update — Phase 6 row to ✅ Complete; v0.4.2 reflected; Next Steps promotes Phase 7 Diebold-Mariano as the immediate next deliverable.

**Phase 7 DM inputs ready as of this notebook:**

- `data/documentation/phase6_step1_arima_forecast.csv` (340 rows)
- `data/documentation/phase6_step2_s6_var_oos_forecasts.csv` (~4,360 rows)
- `data/documentation/phase6_step3_s4_ridge_oos_forecasts.csv` (1,104 rows)

Pre-committed Phase 7 directives:

- USA dual-form DM primary comparison uses Ridge first_diff secondary (D-071 resolution).
- HAC-robust sensitivity recommended alongside standard squared-error loss (D-051).
- Relative-win / absolute-difficulty intellectual honesty framing (D-070) preserved throughout the DM report.

---

*Phase 6 Step 3 complete — 7 sub-steps, 11 decisions (D-064..D-074), 15 audit CSVs, 8 portfolio figures. `src/` v0.4.2 (v0.5.0 reserved for Phase 6 closure module assembly). Next: Phase 7 Diebold-Mariano pairwise forecast comparison across ARIMA / VAR / Ridge at h ∈ {1, 3, 6, 12} months for all four countries.*
